# Similarity-aware Label Smoothing

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders

## Hyperparams

In [24]:
dataset = "cifar100"
lr = 5e-4
batch_size = 128
epochs = 20
beta = 4 if dataset == "imagenet" else 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Models

In [25]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        self.fc1   = nn.Linear(64*7*7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


### CIFAR-100

In [26]:
def RESNET18():
    model = models.resnet18(weights = None)

    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)

    return model

## VAE

### VAE Structure

In [27]:
class ConvVAE(nn.Module):
    def __init__(self, img_size=32, latent_dim=256):
        super().__init__()
        self.img_size = img_size
        self.latent_dim = latent_dim

        # ---------- Encoder ----------
        self.enc = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),  # -> img/2
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),# -> img/4
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),# -> img/8
            nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1),# -> img/16
            nn.ReLU()
        )

        feat_dim = (img_size // 16)
        feat_dim = feat_dim * feat_dim * 512

        self.mu = nn.Linear(feat_dim, latent_dim)
        self.logvar = nn.Linear(feat_dim, latent_dim)

        # ---------- Decoder ----------
        self.fc_dec = nn.Linear(latent_dim, feat_dim)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(h.size(0), -1)
        mu = self.mu(h)
        logvar = self.logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(h.size(0), 512, self.img_size//16, self.img_size//16)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_rec = self.decode(z)
        return x_rec, mu, logvar, z


### VAE Utils

In [28]:
def reconstruction_loss(x, x_hat):
  return F.mse_loss(x_hat, x, reduction='sum')

def kl_divergence(mu, logvar):
  return 0.5 * torch.sum(mu.pow(2) + logvar.exp() - logvar - 1)

def vae_loss(x, x_hat, mu, logvar, beta=1.0):
  batch_size = x.shape[0]
  recon_loss = reconstruction_loss(x, x_hat) / batch_size
  kld = kl_divergence(mu, logvar) / batch_size

  total_loss = recon_loss + beta * kld
  return total_loss, recon_loss, kld

### VAE Training

## Training Utils

### Accuracy Measure

In [29]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            y_expand = y.view(-1, 1).expand_as(pred)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [30]:
def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

## Training Loop

In [31]:
def train(model, train_loader, test_loader, optimizer=None, scheduler=None, num_classes=100, epochs=10, epsilon=0.1):
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = uniform_smooth_labels(y, num_classes=num_classes, epsilon=epsilon)

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

## RUN Experiments

In [32]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

model = RESNET18().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_classes=100,
    epochs=10,
    epsilon=0.1,
)

/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


KeyboardInterrupt: 